## Project 1 

The goal of the first project is to do some wrangling, EDA, and visualization, and generate sequences of values. We will focus on:

- CDC National Health and Nutritional Examination Survey (NHANES, 1999-2000): https://wwwn.cdc.gov/nchs/nhanes/continuousnhanes/default.aspx?BeginYear=1999
- CDC Linked Mortality File (LMF, 1999-2000): https://www.cdc.gov/nchs/data-linkage/mortality-public.htm

NHANES is a rich panel dataset on health and behavior, collected bi-yearly from around 1999 to now. We will focus on the 1999 wave, because that has the largest follow-up window, providing us with the richest mortality data. The mortality data is provided by the CDC Linked Mortality File. 

The purpose of the project is to use $k$-NN to predict who dies (hard or soft classification) and how long they live (regression).

### Part 1: Wrangling and EDA (40/100 pts)

First, go to the NHANES and LMF web sites and familiarize yourself with the data sources. Download codebooks. Think about what resources are available. The CDC Linked Mortality File is somewhat of a pain to work with, so I have pre-cleaned it for you. It is available at httts://github.com/ds4e/undergraduate_ml_assignments in the data folder, as `lmf_parsed.cav`. From the CDC LMF web page, get the SAS program to load the data; it is the real codebook.

Second, download the demographic data for the 1999--2000 wave from the NHANES page. You can use the following code chunk to merge the LMF and DEMO data:

``` python
import pandas as pd
mdf = pd.read_csv('linked_mortality_file_1999_2000.csv') # Load mortality file
print( mdf.head() )
gdf = pd.read_sas("DEMO.xpt", format="xport") # Load demographics file
print( gdf.head() )
df = gdf.merge(mdf, on="SEQN", how="inner") # Merge mortality and demographics on SEQN variable
```

Third, the variables `ELIGSTAT`, `MORTSTAT`, `PERMTH_INT`, and `RIDAGEEX` are particularly important. Look them up in the documentation and clearly describe them. (5/100 pts.)

Second, the goal of the project is to use whatever demographic, behavioral, and health data you like to predict mortality (`MORTSTAT`) and life expectancy (`PERMTH_INT`). Go to the NHANES 1999--2000 web page and select your data and download it. Clearly explain your rationale for selecting these data. Use `.merge` to combine your data into one complete dataframe. Document missing values. (5/100 pts)

Third, do basic EDA and visualization of the key variables. Are any important variables skewed? Are there outliers? How correlated are pairs of variables? Do pairs of categorical variables exhibit interesting patterns in contingency tables? Provide a clear discussion and examination of the data and the variables you are interested in using. (20/100 pts)


### Part 2: $k$-NN classification/regression, write-up (50/100 pts)

Submit a notebook that clearly addresses the following, using code and markdown chunks:

**1. Describe the data, particularly what an observation is and whether there are any missing data that might impact your analysis. Who collected the data and why? What known limitations are there to analysis? (10/100 pts)**

The data comes from the CDC NHANES 1999–2000 survey merged with the CDC Linked Mortality File (LMF), which links health survey participants to mortality outcomes over time. 
Each observation represents one individual participant identified by a unique ID (SEQN). The dataset combines demographic information, smoking behavior, and mortality variables.
There are missing values in the smoking variables due to non-response, refused or unknown answers, and because some smoking questions do not apply to non-smokers. These missing values may reduce sample size and potentially bias results if missingness is related to smoking behavior. The data were collected by the CDC to monitor health behaviors and study how risk factors relate to long-term health outcomes and mortality. Key limitations include self-reported smoking data, the observational nature of the dataset (no causal conclusions), missing values, and the fact that only a subset of possible predictors is included.

**2. Describe the variables you selected to predict mortality and life expectancy, and the rationale behind them. Analyze your variables using describe tables, kernel densities, scatter plots, and conditional kernel densities. Are there any patterns of interest to notice? (10/100 pts)**

The variables selected to predict mortality and life expectancy focus on smoking behavior:
- SMD090 — average cigarettes smoked per day
- SMQ040 — current smoking status
- SMD030 — age started smoking regularly

These variables were chosen because smoking behavior is strongly linked to health risks and mortality. Describe tables show more non-smokers than smokers and many zero values for cigarettes per day. Kernel density plots show that age started smoking is slightly right-skewed, while average cigarettes per day is strongly right-skewed with a long tail of heavy smokers. Conditional densities highlight a large concentration at zero cigarettes for non-smokers. Scatter plots show no strong relationship between age started smoking and cigarettes per day, though there is a slight tendency for earlier smokers to smoke more. Overall, the data show skewed distributions, outliers, and substantial variation in smoking behavior that may help explain differences in mortality and life expectancy.

**3. Using your variables to predict mortality using a $k$-Nearest Neighbor Classifier. Analyze its performance and explain clearly how you select $k$. (10/100 pts)**

(See all of this work in the "collect_data.ibynb" section of this repo)

To predict mortality, we built a k-Nearest Neighbor classifier using smoking behavior variables: current smoking status, average cigarettes per day, and age started smoking. We removed missing values and split the data into training and test sets using a stratified split to preserve the proportion of deaths and survivors. Because kNN relies on distance calculations, we standardized the predictor variables before fitting the model.

To select k, we evaluated values from 1 to 25 and plotted test accuracy against k. The highest test accuracy occurred at k = 4, so we selected this value as the optimal k. Smaller values of k can overfit by relying too heavily on local noise, while larger values smooth predictions too much and may underfit. Choosing k = 4 balanced this tradeoff and produced the best performance on unseen data.

The final model achieved an accuracy of approximately 0.586 (58.6%). The confusion matrix shows that the model predicts survival (class 0) relatively well, with a recall of 0.84, correctly identifying 315 out of 375 survivors. However, it performs poorly at identifying deaths (class 1), with a recall of only 0.23, correctly identifying 60 out of 265 deaths. This indicates the model struggles to detect mortality cases and tends to predict survival more often.

Overall, smoking behavior contains some predictive information about mortality, but the model’s moderate accuracy and low recall for deaths suggest that mortality is influenced by many additional health and demographic factors beyond smoking alone.

**4. Using your variables to predict life expectancy using a $k$-Nearest Neighbor Regressor. Analyze its performance and explain clearly how you select $k$. (10/100 pts)**

**5. Describe how your model could be used for health interventions based on patient characteristics. Are there any limitations or risks to consider? (10/100 pts)**

## Submission (10/100 pts)

Submit your work in a well-organized GitHub repo, where the code is appropriately commented and all members of the group have made significant contributions to the commit history. (10/100 pts)